# Multi-Algorithm PCB Defect Inspection: Authoritative Validation Benchmark

This notebook presents the **head-to-head validation comparison** across all 4 defect inspection paradigms on the **139 validation images** (`data/dataset_split.csv`):

1. **Otsu's Global Thresholding** (Intensity difference + morphology)
2. **Template Matching** (Multi-scale normalized cross-correlation)
3. **Canny Edge Detection** (Gradient difference + closing)
4. **ORB Feature Matching** (Keypoint alignment & spatial descriptor displacement)

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'validation' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Smart auto-run fallback: auto-compute benchmark if outputs/metrics CSV is missing
res_csv = PROJECT_ROOT / 'outputs' / 'metrics' / 'essential_validation_comparison.csv'
if not res_csv.exists():
    print('⚠️ Validation metrics CSV not found. Auto-running validation benchmark on 139 validation images...')
    from scripts.run_essential_validation import main as run_benchmark
    run_benchmark()
    print('✅ Benchmark successfully generated!')

df = pd.read_csv(res_csv)
display(df)


## 1. Best-in-Class Comparison Table (Top-1 Per Algorithm)

In [ ]:
best_per_algo = df.sort_values('f1_score', ascending=False).groupby('algorithm', as_index=False).first()
best_per_algo = best_per_algo.sort_values('f1_score', ascending=False)
display(best_per_algo[['algorithm', 'combination_id', 'description', 'precision', 'recall', 'f1_score', 'mean_runtime_ms']])


## 2. Comparative Benchmark Charts: F1-Score & Latency

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']
ax1.bar(best_per_algo['algorithm'], best_per_algo['f1_score'], color=colors)
ax1.set_ylabel('Validation F1-Score (IoU >= 0.50)', fontsize=11)
ax1.set_title('Top-1 Validation F1-Score by Algorithm', fontsize=12)
ax1.set_ylim(0, 1.0)
for i, v in enumerate(best_per_algo['f1_score']):
    ax1.text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

ax2.bar(best_per_algo['algorithm'], best_per_algo['mean_runtime_ms'], color=colors)
ax2.set_ylabel('Mean Runtime per Image (ms)', fontsize=11)
ax2.set_title('Inference Latency Comparison (Lower is Faster)', fontsize=12)
for i, v in enumerate(best_per_algo['mean_runtime_ms']):
    ax2.text(i, v + 5, f'{v:.1f}ms', ha='center')

plt.tight_layout()
plt.show()
